# Model output metrics

Tables and figures for the 2-stage XGBoost model and the three reference
baselines, at ADM1 level, which is where impact is reported.

Run order: coverage first. The metrics below are only comparable across models
when they are scored on the same administrative units.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.evaluation.bootstrap import bootstrap_binary_metrics
from visualization.metrics_overview import (CATEGORY_LABELS, MODEL_LABELS,
                                            MODEL_ORDER, STRATEGIES,
                                            absolute_errors, categorise,
                                            confusion_counts, coverage,
                                            load_all, restrict_to_common)

STRATEGY = "global"   # global | walk_forward | geo_constrained
LEVEL = "adm1"
EVENT_COL = "sid"     # resample cyclones, not country-event records
N_BOOT = 1000

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

models = load_all(strategy=STRATEGY, level=LEVEL)
print(f"loaded: {sorted(models)}")
if not models:
    raise SystemExit(f"no outputs for strategy {STRATEGY!r} at level {LEVEL!r}")

## 1. Coverage

`shared_units` is the number of (event, ADM1) pairs every loaded model scored.
Where a model scores fewer, the comparison is restricted to the shared set
before any metric is computed.

In [ ]:
cov = coverage(models)
display(cov)

if len(models) > 1 and "only_this_model" in cov and cov["only_this_model"].sum():
    print("models differ in coverage; restricting to shared units")
    models = restrict_to_common(models)
    display(coverage(models))

## 2. Binary metrics with cyclone-clustered confidence intervals

Two thresholds: any reported impact, and impact above 15% of an ADM1 unit's
population. Intervals come from resampling cyclones, so events of the same
storm stay together.

In [ ]:
rows = []
for name in MODEL_ORDER:
    if name not in models:
        continue
    m = bootstrap_binary_metrics(
        models[name], event_col=EVENT_COL, thresholds=(0, 15),
        model_name=MODEL_LABELS.get(name, name), n_boot=N_BOOT, random_state=42)
    rows.append(m)

metrics = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
point_cols = [c for c in ["model", "threshold", "precision", "recall",
                          "specificity", "accuracy", "f1", "csi"]
              if c in metrics.columns]
display(metrics[point_cols].round(4))

In [ ]:
def with_ci(df, metric, decimals=3):
    lo, hi = f"{metric}_ci_low", f"{metric}_ci_high"
    if lo not in df.columns:
        return df[metric].round(decimals).astype(str)
    return (df[metric].round(decimals).astype(str) + " ("
            + df[lo].round(decimals).astype(str) + ", "
            + df[hi].round(decimals).astype(str) + ")")

if not metrics.empty:
    table = pd.DataFrame({"model": metrics["model"], "threshold": metrics["threshold"]})
    for m in ["precision", "recall", "f1", "csi", "accuracy"]:
        if m in metrics.columns:
            table[m] = with_ci(metrics, m)
    display(table)

## 3. Categorical confusion matrices

Rows are reported, columns predicted, in the three impact categories. Cells are
row-normalised, so each row shows how units of that reported class were
classified.

In [ ]:
present = [n for n in MODEL_ORDER if n in models]
fig, axes = plt.subplots(1, len(present), figsize=(4.2 * len(present), 4),
                         squeeze=False)

for ax, name in zip(axes[0], present):
    cm = confusion_counts(models[name])
    norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{norm[i, j]:.2f}\n({cm[i, j]:,})", ha="center",
                    va="center", fontsize=8,
                    color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xticks(range(3), CATEGORY_LABELS, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(3), CATEGORY_LABELS, fontsize=8)
    ax.set_title(MODEL_LABELS.get(name, name), fontsize=10)
    ax.set_xlabel("predicted")
axes[0][0].set_ylabel("reported")
fig.colorbar(im, ax=axes[0], fraction=0.025, label="share of reported class")
plt.show()

## 4. Absolute error

Left: the full distribution. Right: the same data restricted to the lower
range, where most units sit.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
data = [absolute_errors(models[n]) for n in present]
labels = [MODEL_LABELS.get(n, n) for n in present]

for ax, cap, title in ((axes[0], None, "all units"),
                       (axes[1], 10, "absolute error <= 10 points")):
    d = data if cap is None else [x[x <= cap] for x in data]
    parts = ax.violinplot(d, showmedians=True, widths=0.8)
    for pc in parts["bodies"]:
        pc.set_alpha(0.65)
    ax.set_xticks(range(1, len(labels) + 1), labels, rotation=20, ha="right",
                  fontsize=9)
    ax.set_ylabel("|predicted - reported| (percentage points)")
    ax.set_title(title, fontsize=10)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "model": labels,
    "MAE": [x.mean() for x in data],
    "median AE": [np.median(x) for x in data],
    "p90 AE": [np.percentile(x, 90) for x in data],
    "RMSE": [np.sqrt((x ** 2).mean()) for x in data],
}).round(3)
display(summary)

## 5. Cross-validation strategies

The same model under each strategy. `walk_forward` trains only on cyclones
preceding the test cyclone, so the earliest events in the record cannot be
scored and its event count is lower by construction.

In [ ]:
rows = []
for strategy in STRATEGIES:
    got = load_all(strategy=strategy, level=LEVEL, models=["2-stage-XGBoost"])
    if not got:
        print(f"{strategy}: not scored yet")
        continue
    df = got["2-stage-XGBoost"]
    m = bootstrap_binary_metrics(
        df, event_col=EVENT_COL, thresholds=(0, 15),
        model_name=strategy, n_boot=N_BOOT, random_state=42)
    m["events"] = df["DisNo."].nunique()
    m["units"] = len(df)
    rows.append(m)

if rows:
    by_strategy = pd.concat(rows, ignore_index=True)
    cols = [c for c in ["model", "threshold", "events", "units", "precision",
                        "recall", "f1", "csi"] if c in by_strategy.columns]
    display(by_strategy[cols].round(4))

In [ ]:
if rows:
    fig, ax = plt.subplots(figsize=(7, 4))
    width = 0.35
    for k, thr in enumerate((0, 15)):
        sub = by_strategy[by_strategy["threshold"] == thr]
        x = np.arange(len(sub))
        lo = (sub["f1"] - sub["f1_ci_low"]).to_numpy() if "f1_ci_low" in sub else None
        hi = (sub["f1_ci_high"] - sub["f1"]).to_numpy() if "f1_ci_high" in sub else None
        ax.bar(x + k * width, sub["f1"], width,
               yerr=None if lo is None else [lo, hi], capsize=3,
               label=f"threshold {thr}%")
        ax.set_xticks(x + width / 2, sub["model"], rotation=15)
    ax.set_ylabel("F1 (ADM1)")
    ax.set_title("F1 by cross-validation strategy")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()